In [1]:
import os, time, requests
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

cities = {
    "Amsterdam": {"lat": 52.3676, "lon": 4.9041,  "country": "NLD"},
    "Barcelona": {"lat": 41.3874, "lon": 2.1686,  "country": "ESP"},
    "Berlin":    {"lat": 52.5200, "lon": 13.4050, "country": "DEU"},
    "Columbus":  {"lat": 39.9612, "lon": -82.9988, "country": "USA"},
}

In [2]:
START, END = "2025-09-17", "2026-09-19"
rows = []
for city, c in cities.items():
    params = {
        "latitude": c["lat"], "longitude": c["lon"],
        "start_date": START, "end_date": END,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "auto",
    }
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
    r.raise_for_status()
    d = r.json()["daily"]
    rows.append(pd.DataFrame({
        "city": city, "date": d["time"],
        "temp_max": d["temperature_2m_max"],
        "temp_min": d["temperature_2m_min"],
        "precipitation": d["precipitation_sum"],
    }))
    time.sleep(1)

weather = pd.concat(rows, ignore_index=True)
weather["note"] = "reference weather: same calendar days one year earlier"
print(weather.groupby("city").size())
weather.head()

city
Amsterdam    368
Barcelona    368
Berlin       368
Columbus     368
dtype: int64


,city,date,temp_max,temp_min,precipitation,note
0,Amsterdam,2025-09-17,17.4,13.1,9.7,reference weather: same calendar days one year...
1,Amsterdam,2025-09-18,20.5,17.0,5.5,reference weather: same calendar days one year...
2,Amsterdam,2025-09-19,24.1,14.9,0.0,reference weather: same calendar days one year...
3,Amsterdam,2025-09-20,22.2,17.2,1.2,reference weather: same calendar days one year...
4,Amsterdam,2025-09-21,19.4,11.4,0.2,reference weather: same calendar days one year...


In [3]:
weather.to_sql("raw_weather", engine, if_exists="replace", index=False)
with engine.connect() as conn:
    print(conn.execute(text("SELECT city, COUNT(*), MIN(date), MAX(date) FROM raw_weather GROUP BY city")).fetchall())

[('Amsterdam', 368, '2025-09-17', '2026-09-19'), ('Barcelona', 368, '2025-09-17', '2026-09-19'), ('Berlin', 368, '2025-09-17', '2026-09-19'), ('Columbus', 368, '2025-09-17', '2026-09-19')]
